# 🚀 VQA Vector Search - Azure ML Setup

## Quick Start (Serverless GPU - Pay Only When Running)

### 1. Create Azure ML Workspace
```
Azure Portal > Create Resource > Machine Learning > Create
```

### 2. Create Compute Cluster (Scales to Zero!)
```
Azure ML Studio > Compute > Compute clusters > New
- Name: vqa-gpu-cluster
- VM: NC4as_T4_v3 (cheapest GPU, ~$0.53/hr)
- Min nodes: 0 (scales to zero when idle!)
- Max nodes: 1
- Idle time: 120 seconds
```

### 3. Set Environment Variable
```
Azure ML Studio > Compute > Your Cluster > Environment variables
- Name: COSMOS_CONNECTION_STRING
- Value: your-cosmos-connection-string
```

### 4. Run This Notebook
- Data downloads to compute's local storage (~6GB)
- Embeddings generated and uploaded to Cosmos DB
- When done, compute scales to zero → data auto-deleted

**💰 Total Cost: ~$2-5 for one-time processing**

---
# 🔧 Phase 0: Setup & Configuration

In [ ]:
# Cell 1: Install Required Packages
# Run this cell first - may take 2-3 minutes

!pip install --upgrade pip
!pip install transformers>=4.35.0
!pip install torch>=2.0.0 torchvision>=0.15.0
!pip install pymongo[srv]>=4.6.0
!pip install Pillow>=10.0.0
!pip install tqdm>=4.66.0
!pip install requests>=2.31.0
!pip install numpy>=1.24.0
!pip install pandas>=2.0.0

print("\n✓ All packages installed!")

In [ ]:
# Cell 2: Import Libraries & Check GPU

import os
import json
import time
import pickle
import requests
import zipfile
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple, Optional
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from transformers import ViltProcessor, ViltModel
from pymongo import MongoClient
from pymongo.errors import BulkWriteError

# Check GPU availability
print("=" * 60)
print("GPU Configuration")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = torch.device("cuda")
else:
    print("⚠️ No GPU detected! Embedding generation will be very slow.")
    print("   Consider using Azure ML with GPU compute.")
    DEVICE = torch.device("cpu")

print(f"\n✓ Using device: {DEVICE}")

In [ ]:
# Cell 3: Configuration Settings
import os

# ============================================
# 🔧 CONFIGURATION - MODIFY THESE VALUES
# ============================================

# Azure Cosmos DB Settings
# Account name: vqa-vector-db
# Get connection string from: Azure Portal > Cosmos DB > Settings > Connection strings

# Option 1: Set as environment variable (recommended for Azure ML)
#   - In Azure ML: Compute > Environment variables
#   - Or: export COSMOS_CONNECTION_STRING="your-connection-string"
# Option 2: Paste directly here (for local testing only)
COSMOS_CONNECTION_STRING = os.getenv("COSMOS_CONNECTION_STRING", "")

if not COSMOS_CONNECTION_STRING:
    # Fallback for local testing - DELETE before committing!
    COSMOS_CONNECTION_STRING = "YOUR_CONNECTION_STRING_HERE"
    print("⚠️ Using hardcoded connection string - set COSMOS_CONNECTION_STRING env var for production!")

# Database and Collection names (inside Cosmos DB account "vqa-vector-db")
DATABASE_NAME = "vqa_vectors"
COLLECTION_NAME = "embeddings"

# Dataset settings - downloads to compute's local storage (auto-deleted when compute stops)
DATA_DIR = Path("./vqa_data")  # Local temp storage on compute
USE_VALIDATION_SET = True  # True = smaller set (~214k), False = training set (~443k)
SAMPLE_FRACTION = 0.33  # Use 1/3 of dataset for faster processing

# Processing settings
BATCH_SIZE = 32  # Reduce if GPU memory issues (16 for 8GB GPU)
CHECKPOINT_EVERY = 1000  # Save progress every N samples
MAX_RETRIES = 3  # Retry failed operations

# Embedding settings
EMBEDDING_DIM = 768  # ViLT hidden size
MAX_QUESTION_LENGTH = 40  # Truncate long questions

# ============================================

# Create data directory
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Configuration loaded")
print(f"  Cosmos DB account: vqa-vector-db")
print(f"  Database: {DATABASE_NAME}")
print(f"  Collection: {COLLECTION_NAME}")
print(f"  Data directory: {DATA_DIR.absolute()}")
print(f"  Dataset: {'Validation' if USE_VALIDATION_SET else 'Training'} ({SAMPLE_FRACTION*100:.0f}%)")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Embedding dimension: {EMBEDDING_DIM}")
print(f"\n💡 Data will be downloaded to compute's local storage")
print(f"   No Azure Blob Storage needed - data auto-deleted when compute stops")

In [ ]:
# Cell 4: Checkpoint System

class CheckpointManager:
    """Manage checkpoints to resume interrupted processing."""
    
    def __init__(self, checkpoint_dir: Path = None):
        self.checkpoint_dir = checkpoint_dir or DATA_DIR / "checkpoints"
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.checkpoint_file = self.checkpoint_dir / "progress.json"
    
    def save(self, phase: str, progress: dict):
        """Save checkpoint."""
        checkpoint = {
            "phase": phase,
            "progress": progress,
            "timestamp": datetime.now().isoformat()
        }
        with open(self.checkpoint_file, 'w') as f:
            json.dump(checkpoint, f, indent=2)
        print(f"💾 Checkpoint saved: {phase} - {progress}")
    
    def load(self) -> Optional[dict]:
        """Load checkpoint if exists."""
        if self.checkpoint_file.exists():
            with open(self.checkpoint_file, 'r') as f:
                checkpoint = json.load(f)
            print(f"📂 Loaded checkpoint: {checkpoint['phase']} from {checkpoint['timestamp']}")
            return checkpoint
        return None
    
    def clear(self):
        """Clear checkpoint."""
        if self.checkpoint_file.exists():
            self.checkpoint_file.unlink()
            print("🗑️ Checkpoint cleared")

checkpoint_mgr = CheckpointManager()
print("✓ Checkpoint system initialized")

---
# 📥 Phase 1: Download VQAv2 Dataset

In [ ]:
# Cell 5: Download Helper Functions

def download_file(url: str, destination: Path, description: str = None) -> bool:
    """Download file with progress bar and resume support."""
    destination = Path(destination)
    
    # Skip if already exists
    if destination.exists():
        print(f"⏭️ Already exists: {destination.name}")
        return True
    
    destination.parent.mkdir(parents=True, exist_ok=True)
    
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()
        
        total_size = int(response.headers.get('content-length', 0))
        
        with open(destination, 'wb') as f:
            with tqdm(total=total_size, unit='B', unit_scale=True, 
                     desc=description or destination.name) as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
                    pbar.update(len(chunk))
        
        print(f"✓ Downloaded: {destination.name}")
        return True
        
    except Exception as e:
        print(f"✗ Download failed: {e}")
        if destination.exists():
            destination.unlink()
        return False

def extract_zip(zip_path: Path, extract_to: Path) -> bool:
    """Extract zip file."""
    try:
        print(f"📦 Extracting: {zip_path.name}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"✓ Extracted to: {extract_to}")
        return True
    except Exception as e:
        print(f"✗ Extraction failed: {e}")
        return False

print("✓ Download helpers ready")

In [ ]:
# Cell 6: Download VQAv2 Annotations and Questions

# URLs for VQAv2 dataset
VQA_URLS = {
    "train_annotations": "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip",
    "val_annotations": "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Val_mscoco.zip",
    "train_questions": "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip",
    "val_questions": "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Val_mscoco.zip",
}

# COCO Image URLs
COCO_URLS = {
    "train_images": "http://images.cocodataset.org/zips/train2014.zip",  # 13GB
    "val_images": "http://images.cocodataset.org/zips/val2014.zip",      # 6GB
}

print("=" * 60)
print("Downloading VQAv2 Annotations & Questions")
print("=" * 60)

# Download annotations
if USE_VALIDATION_SET:
    ann_url = VQA_URLS["val_annotations"]
    que_url = VQA_URLS["val_questions"]
else:
    ann_url = VQA_URLS["train_annotations"]
    que_url = VQA_URLS["train_questions"]

ann_zip = DATA_DIR / "annotations.zip"
que_zip = DATA_DIR / "questions.zip"

download_file(ann_url, ann_zip, "Annotations")
download_file(que_url, que_zip, "Questions")

# Extract
extract_zip(ann_zip, DATA_DIR)
extract_zip(que_zip, DATA_DIR)

print("\n✓ Annotations and questions ready!")

In [ ]:
# Cell 7: Download COCO Images
# ⚠️ This downloads ~6-13GB of images. Skip if you already have them.

print("=" * 60)
print("Downloading COCO Images")
print("=" * 60)
print("⚠️ This will download ~6GB (val) or ~13GB (train) of images")
print("   Skip this cell if you already have the images.\n")

if USE_VALIDATION_SET:
    images_url = COCO_URLS["val_images"]
    images_zip = DATA_DIR / "val2014.zip"
    images_dir = DATA_DIR / "val2014"
else:
    images_url = COCO_URLS["train_images"]
    images_zip = DATA_DIR / "train2014.zip"
    images_dir = DATA_DIR / "train2014"

# Check if images already exist
if images_dir.exists() and len(list(images_dir.glob("*.jpg"))) > 1000:
    print(f"⏭️ Images already exist in {images_dir}")
    print(f"   Found {len(list(images_dir.glob('*.jpg')))} images")
else:
    download_file(images_url, images_zip, "COCO Images")
    extract_zip(images_zip, DATA_DIR)

print("\n✓ Images ready!")

---
# 🔄 Phase 2: Data Preparation

In [ ]:
# Cell 8: Load VQAv2 Dataset

def load_vqa_data(data_dir: Path, use_validation: bool = True) -> Tuple[dict, dict]:
    """Load VQAv2 annotations and questions."""
    
    split = "val" if use_validation else "train"
    
    # Find files
    ann_file = data_dir / f"v2_mscoco_{split}2014_annotations.json"
    que_file = data_dir / f"v2_OpenEnded_mscoco_{split}2014_questions.json"
    
    print(f"Loading {split} dataset...")
    
    # Load annotations
    with open(ann_file, 'r') as f:
        annotations = json.load(f)
    print(f"  ✓ Loaded {len(annotations['annotations'])} annotations")
    
    # Load questions
    with open(que_file, 'r') as f:
        questions = json.load(f)
    print(f"  ✓ Loaded {len(questions['questions'])} questions")
    
    return annotations, questions

# Load data
annotations_data, questions_data = load_vqa_data(DATA_DIR, USE_VALIDATION_SET)

# Create lookup dictionaries
annotations_lookup = {a['question_id']: a for a in annotations_data['annotations']}
questions_lookup = {q['question_id']: q for q in questions_data['questions']}

print(f"\n✓ Dataset loaded: {len(questions_lookup)} QA pairs")

In [ ]:
# Cell 9: Prepare Dataset for Processing

def get_most_common_answer(annotation: dict) -> Tuple[str, float]:
    """Get most common answer and agreement score."""
    answers = [a['answer'] for a in annotation['answers']]
    counter = Counter(answers)
    most_common = counter.most_common(1)[0]
    return most_common[0], most_common[1] / len(answers)

def get_question_type(question: str) -> str:
    """Classify question type."""
    q = question.lower().strip()
    
    if q.startswith("how many"):
        return "count"
    elif q.startswith("what color"):
        return "color"
    elif q.startswith("what is") or q.startswith("what are"):
        return "what"
    elif q.startswith("where"):
        return "location"
    elif q.startswith("who"):
        return "person"
    elif q.startswith(("is ", "are ", "does ", "do ", "can ", "could ", "was ", "were ")):
        return "yes/no"
    else:
        return "other"

def prepare_samples(questions_lookup: dict, annotations_lookup: dict, 
                   images_dir: Path, sample_fraction: float = 1.0) -> List[dict]:
    """Prepare samples for processing."""
    
    samples = []
    split = "val2014" if USE_VALIDATION_SET else "train2014"
    
    for qid, question_data in tqdm(questions_lookup.items(), desc="Preparing samples"):
        annotation = annotations_lookup.get(qid)
        if not annotation:
            continue
        
        image_id = question_data['image_id']
        image_filename = f"COCO_{split}_{image_id:012d}.jpg"
        image_path = images_dir / image_filename
        
        # Skip if image doesn't exist
        if not image_path.exists():
            continue
        
        answer, confidence = get_most_common_answer(annotation)
        
        samples.append({
            "question_id": qid,
            "image_id": image_id,
            "question": question_data['question'],
            "answer": answer,
            "answer_confidence": confidence,
            "question_type": get_question_type(question_data['question']),
            "image_path": str(image_path),
            "answer_type": annotation.get('answer_type', 'unknown')
        })
    
    # Sample fraction
    if sample_fraction < 1.0:
        np.random.seed(42)  # Reproducibility
        n_samples = int(len(samples) * sample_fraction)
        indices = np.random.choice(len(samples), n_samples, replace=False)
        samples = [samples[i] for i in sorted(indices)]
    
    return samples

# Prepare samples
images_dir = DATA_DIR / ("val2014" if USE_VALIDATION_SET else "train2014")
samples = prepare_samples(questions_lookup, annotations_lookup, images_dir, SAMPLE_FRACTION)

print(f"\n✓ Prepared {len(samples)} samples ({SAMPLE_FRACTION*100:.0f}% of dataset)")

In [ ]:
# Cell 10: Dataset Statistics

print("=" * 60)
print("Dataset Statistics")
print("=" * 60)

# Question type distribution
qtype_dist = Counter([s['question_type'] for s in samples])
print("\n📊 Question Type Distribution:")
for qtype, count in qtype_dist.most_common():
    pct = count / len(samples) * 100
    print(f"  {qtype:12s}: {count:6d} ({pct:5.1f}%)")

# Answer type distribution
atype_dist = Counter([s['answer_type'] for s in samples])
print("\n📊 Answer Type Distribution:")
for atype, count in atype_dist.most_common():
    pct = count / len(samples) * 100
    print(f"  {atype:12s}: {count:6d} ({pct:5.1f}%)")

# Show sample entries
print("\n📝 Sample Entries:")
for i, sample in enumerate(samples[:3]):
    print(f"\n  [{i+1}]")
    print(f"  Q: {sample['question']}")
    print(f"  A: {sample['answer']} (confidence: {sample['answer_confidence']:.0%})")
    print(f"  Type: {sample['question_type']}")

In [ ]:
# Cell 11: Save Prepared Samples (Checkpoint)

samples_file = DATA_DIR / "prepared_samples.pkl"

with open(samples_file, 'wb') as f:
    pickle.dump(samples, f)

checkpoint_mgr.save("data_preparation", {
    "num_samples": len(samples),
    "sample_fraction": SAMPLE_FRACTION,
    "samples_file": str(samples_file)
})

print(f"✓ Saved {len(samples)} samples to {samples_file}")

---
# 🗄️ Phase 3: Azure Cosmos DB Setup

In [ ]:
# Cell 12: Connect to Cosmos DB

def connect_to_cosmos(connection_string: str, database: str, collection: str):
    """Connect to Azure Cosmos DB for MongoDB."""
    try:
        print("Connecting to Cosmos DB...")
        client = MongoClient(connection_string, serverSelectionTimeoutMS=10000)
        
        # Test connection
        client.admin.command('ping')
        print("✓ Connected successfully!")
        
        db = client[database]
        coll = db[collection]
        
        # Get document count
        doc_count = coll.count_documents({})
        print(f"✓ Database: {database}")
        print(f"✓ Collection: {collection}")
        print(f"✓ Existing documents: {doc_count}")
        
        return client, db, coll
        
    except Exception as e:
        print(f"✗ Connection failed: {e}")
        raise

# Connect
cosmos_client, cosmos_db, cosmos_collection = connect_to_cosmos(
    COSMOS_CONNECTION_STRING, DATABASE_NAME, COLLECTION_NAME
)

In [ ]:
# Cell 13: Create Vector Search Index

def create_vector_index(collection, index_name: str = "vqa_vector_index", 
                       dimensions: int = 768):
    """Create vector search index for Cosmos DB."""
    
    # Check if index exists
    existing_indexes = list(collection.list_search_indexes())
    if any(idx.get('name') == index_name for idx in existing_indexes):
        print(f"⏭️ Index '{index_name}' already exists")
        return
    
    # Create vector index
    index_definition = {
        "mappings": {
            "dynamic": True,
            "fields": {
                "embedding": {
                    "type": "knnVector",
                    "dimensions": dimensions,
                    "similarity": "cosine"
                }
            }
        }
    }
    
    try:
        from pymongo.operations import SearchIndexModel
        search_index = SearchIndexModel(
            definition=index_definition,
            name=index_name
        )
        collection.create_search_index(search_index)
        print(f"✓ Created vector index: {index_name}")
        print(f"  Dimensions: {dimensions}")
        print(f"  Similarity: cosine")
    except Exception as e:
        print(f"⚠️ Index creation note: {e}")
        print("  Index may be created automatically on first vector insert")

# Create index (dimensions = 768 for ViLT)
create_vector_index(cosmos_collection, dimensions=EMBEDDING_DIM)

---
# 🧠 Phase 4: ViLT Embedding Generation

In [ ]:
# Cell 14: Load ViLT Model

print("=" * 60)
print("Loading ViLT Model")
print("=" * 60)

# Load ViLT processor and model
print("Loading processor...")
vilt_processor = ViltProcessor.from_pretrained("dandelin/vilt-b32-mlm")

print("Loading model...")
vilt_model = ViltModel.from_pretrained("dandelin/vilt-b32-mlm")
vilt_model = vilt_model.to(DEVICE)
vilt_model.eval()  # Set to evaluation mode

print(f"\n✓ ViLT model loaded on {DEVICE}")
print(f"✓ Embedding dimension: {vilt_model.config.hidden_size}")

In [ ]:
# Cell 15: Embedding Generation Functions

def generate_embedding(image: Image.Image, question: str, 
                      processor, model, device) -> np.ndarray:
    """Generate ViLT embedding for image-question pair."""
    
    # Preprocess
    inputs = processor(image, question, return_tensors="pt", 
                      truncation=True, max_length=MAX_QUESTION_LENGTH)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Generate embedding
    with torch.no_grad():
        outputs = model(**inputs)
        # Use [CLS] token embedding (pooled output)
        embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
    
    # Normalize
    embedding = embedding / np.linalg.norm(embedding)
    
    return embedding.flatten()

def generate_embeddings_batch(samples: List[dict], processor, model, 
                             device, batch_size: int = 32) -> List[np.ndarray]:
    """Generate embeddings for a batch of samples."""
    
    embeddings = []
    
    for i in range(0, len(samples), batch_size):
        batch = samples[i:i + batch_size]
        
        # Load images
        images = []
        questions = []
        valid_indices = []
        
        for j, sample in enumerate(batch):
            try:
                img = Image.open(sample['image_path']).convert('RGB')
                images.append(img)
                questions.append(sample['question'])
                valid_indices.append(j)
            except Exception as e:
                print(f"⚠️ Error loading image: {e}")
        
        if not images:
            embeddings.extend([None] * len(batch))
            continue
        
        # Process batch
        try:
            inputs = processor(images, questions, return_tensors="pt", 
                             padding=True, truncation=True, 
                             max_length=MAX_QUESTION_LENGTH)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = model(**inputs)
                batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            
            # Normalize
            norms = np.linalg.norm(batch_embeddings, axis=1, keepdims=True)
            batch_embeddings = batch_embeddings / norms
            
            # Map back to original indices
            result = [None] * len(batch)
            for idx, emb in zip(valid_indices, batch_embeddings):
                result[idx] = emb
            embeddings.extend(result)
            
        except Exception as e:
            print(f"⚠️ Batch processing error: {e}")
            embeddings.extend([None] * len(batch))
    
    return embeddings

print("✓ Embedding functions ready")

In [ ]:
# Cell 16: Test Embedding Generation

print("Testing embedding generation...")

# Test with first sample
test_sample = samples[0]
test_image = Image.open(test_sample['image_path']).convert('RGB')

start_time = time.time()
test_embedding = generate_embedding(
    test_image, test_sample['question'], 
    vilt_processor, vilt_model, DEVICE
)
elapsed = time.time() - start_time

print(f"\n✓ Test embedding generated")
print(f"  Shape: {test_embedding.shape}")
print(f"  Dtype: {test_embedding.dtype}")
print(f"  Norm: {np.linalg.norm(test_embedding):.4f} (should be ~1.0)")
print(f"  Time: {elapsed:.3f}s")
print(f"\n  Question: {test_sample['question']}")
print(f"  Answer: {test_sample['answer']}")

In [ ]:
# Cell 17: Generate All Embeddings

print("=" * 60)
print(f"Generating Embeddings for {len(samples)} Samples")
print("=" * 60)

# Load checkpoint if exists
embeddings_cache_file = DATA_DIR / "embeddings_cache.pkl"
start_idx = 0
all_embeddings = []

if embeddings_cache_file.exists():
    with open(embeddings_cache_file, 'rb') as f:
        cache = pickle.load(f)
    all_embeddings = cache['embeddings']
    start_idx = cache['last_idx']
    print(f"📂 Resuming from checkpoint: {start_idx}/{len(samples)}")

# Process remaining samples
remaining_samples = samples[start_idx:]
total_batches = (len(remaining_samples) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Processing {len(remaining_samples)} remaining samples in {total_batches} batches...\n")

for batch_idx in tqdm(range(0, len(remaining_samples), BATCH_SIZE), 
                      desc="Generating embeddings", total=total_batches):
    
    batch = remaining_samples[batch_idx:batch_idx + BATCH_SIZE]
    
    # Generate embeddings for batch
    batch_embeddings = generate_embeddings_batch(
        batch, vilt_processor, vilt_model, DEVICE, BATCH_SIZE
    )
    all_embeddings.extend(batch_embeddings)
    
    # Checkpoint
    current_idx = start_idx + batch_idx + len(batch)
    if (batch_idx + BATCH_SIZE) % CHECKPOINT_EVERY < BATCH_SIZE:
        with open(embeddings_cache_file, 'wb') as f:
            pickle.dump({
                'embeddings': all_embeddings,
                'last_idx': current_idx
            }, f)
        print(f"\n💾 Checkpoint: {current_idx}/{len(samples)}")

# Final save
with open(embeddings_cache_file, 'wb') as f:
    pickle.dump({
        'embeddings': all_embeddings,
        'last_idx': len(samples)
    }, f)

print(f"\n✓ Generated {len(all_embeddings)} embeddings")
print(f"✓ Failed: {sum(1 for e in all_embeddings if e is None)}")

---
# ⬆️ Phase 5: Upload to Cosmos DB

In [ ]:
# Cell 18: Prepare Documents for Upload

def prepare_document(sample: dict, embedding: np.ndarray) -> dict:
    """Prepare document for Cosmos DB insertion."""
    
    return {
        "_id": str(sample['question_id']),
        "question_id": sample['question_id'],
        "image_id": sample['image_id'],
        "question": sample['question'],
        "answer": sample['answer'],
        "answer_confidence": sample['answer_confidence'],
        "question_type": sample['question_type'],
        "answer_type": sample['answer_type'],
        "embedding": embedding.tolist(),
        "created_at": datetime.utcnow().isoformat()
    }

# Prepare all documents
documents = []
skipped = 0

for sample, embedding in zip(samples, all_embeddings):
    if embedding is None:
        skipped += 1
        continue
    documents.append(prepare_document(sample, embedding))

print(f"✓ Prepared {len(documents)} documents for upload")
print(f"✓ Skipped {skipped} failed embeddings")

In [ ]:
# Cell 19: Upload Documents to Cosmos DB

def upload_documents(collection, documents: List[dict], batch_size: int = 100):
    """Upload documents to Cosmos DB with retry logic."""
    
    total_uploaded = 0
    total_errors = 0
    
    for i in tqdm(range(0, len(documents), batch_size), desc="Uploading"):
        batch = documents[i:i + batch_size]
        
        for attempt in range(MAX_RETRIES):
            try:
                # Use insert_many with ordered=False to continue on errors
                result = collection.insert_many(batch, ordered=False)
                total_uploaded += len(result.inserted_ids)
                break
                
            except BulkWriteError as bwe:
                # Some documents may have been inserted
                n_inserted = bwe.details.get('nInserted', 0)
                total_uploaded += n_inserted
                
                # Count duplicate key errors (already exist)
                write_errors = bwe.details.get('writeErrors', [])
                duplicates = sum(1 for e in write_errors if e.get('code') == 11000)
                other_errors = len(write_errors) - duplicates
                total_errors += other_errors
                
                if duplicates > 0:
                    pass  # Skip duplicates silently
                break
                
            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    print(f"\n⚠️ Retry {attempt + 1}/{MAX_RETRIES}: {e}")
                    time.sleep(2 ** attempt)  # Exponential backoff
                else:
                    print(f"\n✗ Failed after {MAX_RETRIES} attempts: {e}")
                    total_errors += len(batch)
    
    return total_uploaded, total_errors

print("=" * 60)
print("Uploading to Cosmos DB")
print("=" * 60)

uploaded, errors = upload_documents(cosmos_collection, documents, batch_size=100)

print(f"\n✓ Uploaded: {uploaded} documents")
print(f"✓ Errors: {errors}")
print(f"✓ Total in collection: {cosmos_collection.count_documents({})}")

In [ ]:
# Cell 20: Verify Upload

print("=" * 60)
print("Verifying Upload")
print("=" * 60)

# Count documents
total_docs = cosmos_collection.count_documents({})
print(f"\n📊 Total documents: {total_docs}")

# Check sample document
sample_doc = cosmos_collection.find_one()
if sample_doc:
    print(f"\n📝 Sample Document:")
    print(f"  Question ID: {sample_doc['question_id']}")
    print(f"  Question: {sample_doc['question']}")
    print(f"  Answer: {sample_doc['answer']}")
    print(f"  Embedding dim: {len(sample_doc['embedding'])}")

# Question type distribution
pipeline = [
    {"$group": {"_id": "$question_type", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}
]
print(f"\n📊 Question Type Distribution:")
for doc in cosmos_collection.aggregate(pipeline):
    print(f"  {doc['_id']:12s}: {doc['count']:6d}")

checkpoint_mgr.save("upload_complete", {
    "total_documents": total_docs,
    "timestamp": datetime.now().isoformat()
})

---
# 🔍 Phase 6: Query System

In [ ]:
# Cell 21: Vector Search Function

def vector_search(collection, query_embedding: np.ndarray, k: int = 5,
                 filter_query: dict = None) -> List[dict]:
    """Perform vector similarity search."""
    
    # Build aggregation pipeline
    pipeline = [
        {
            "$search": {
                "cosmosSearch": {
                    "vector": query_embedding.tolist(),
                    "path": "embedding",
                    "k": k
                },
                "returnStoredSource": True
            }
        },
        {
            "$addFields": {
                "similarity_score": {"$meta": "searchScore"}
            }
        },
        {
            "$project": {
                "embedding": 0  # Don't return embedding (too large)
            }
        }
    ]
    
    # Add filter if provided
    if filter_query:
        pipeline.insert(1, {"$match": filter_query})
    
    results = list(collection.aggregate(pipeline))
    return results

print("✓ Vector search function ready")

In [ ]:
# Cell 22: Query Interface

class VQAQuerySystem:
    """Interactive VQA query system."""
    
    def __init__(self, collection, processor, model, device):
        self.collection = collection
        self.processor = processor
        self.model = model
        self.device = device
    
    def query(self, image_path: str, question: str, k: int = 5,
             filter_type: str = None) -> dict:
        """Query the VQA system with an image and question."""
        
        # Load image
        image = Image.open(image_path).convert('RGB')
        
        # Generate embedding
        start_time = time.time()
        query_embedding = generate_embedding(
            image, question, self.processor, self.model, self.device
        )
        embed_time = time.time() - start_time
        
        # Search
        filter_query = {"question_type": filter_type} if filter_type else None
        start_time = time.time()
        results = vector_search(self.collection, query_embedding, k, filter_query)
        search_time = time.time() - start_time
        
        # Get predicted answer (majority vote)
        if results:
            answers = [r['answer'] for r in results]
            counter = Counter(answers)
            predicted_answer = counter.most_common(1)[0][0]
            confidence = counter.most_common(1)[0][1] / len(answers)
        else:
            predicted_answer = "unknown"
            confidence = 0.0
        
        return {
            "question": question,
            "predicted_answer": predicted_answer,
            "confidence": confidence,
            "similar_examples": results,
            "embedding_time_ms": embed_time * 1000,
            "search_time_ms": search_time * 1000
        }
    
    def display_results(self, result: dict):
        """Display query results nicely."""
        print("\n" + "=" * 60)
        print("🔍 QUERY RESULTS")
        print("=" * 60)
        print(f"\n❓ Question: {result['question']}")
        print(f"\n🎯 Predicted Answer: {result['predicted_answer']}")
        print(f"   Confidence: {result['confidence']:.0%}")
        print(f"\n⏱️ Embedding time: {result['embedding_time_ms']:.1f}ms")
        print(f"   Search time: {result['search_time_ms']:.1f}ms")
        
        print(f"\n📋 Similar Examples:")
        for i, ex in enumerate(result['similar_examples'][:5], 1):
            score = ex.get('similarity_score', 0)
            print(f"\n   [{i}] Score: {score:.3f}")
            print(f"       Q: {ex['question']}")
            print(f"       A: {ex['answer']}")

# Initialize query system
query_system = VQAQuerySystem(
    cosmos_collection, vilt_processor, vilt_model, DEVICE
)

print("✓ Query system initialized")

In [ ]:
# Cell 23: Test Query

# Test with a sample from the dataset
test_sample = samples[100]  # Pick any sample

print(f"Testing with image: {test_sample['image_path']}")
print(f"Ground truth answer: {test_sample['answer']}")

result = query_system.query(
    image_path=test_sample['image_path'],
    question=test_sample['question'],
    k=5
)

query_system.display_results(result)

# Check if prediction matches
match = result['predicted_answer'].lower() == test_sample['answer'].lower()
print(f"\n{'✓ CORRECT!' if match else '✗ INCORRECT'}")
print(f"Ground truth: {test_sample['answer']}")

In [ ]:
# Cell 24: Interactive Query (Try Your Own Questions!)

# ============================================
# 🎮 TRY YOUR OWN QUERIES HERE
# ============================================

# Option 1: Use an image from the dataset
test_image_path = samples[0]['image_path']  # Change index to try different images

# Option 2: Upload your own image (uncomment below)
# test_image_path = "path/to/your/image.jpg"

# Your question
my_question = "What color is the object?"  # Change this!

# Run query
result = query_system.query(
    image_path=test_image_path,
    question=my_question,
    k=5
)

query_system.display_results(result)

# Display the image
from IPython.display import display
img = Image.open(test_image_path)
img.thumbnail((400, 400))
display(img)

---
# 📊 Phase 7: Evaluation

In [ ]:
# Cell 25: Evaluation Functions

def evaluate_vqa_accuracy(query_system, test_samples: List[dict], 
                         k: int = 5) -> dict:
    """Evaluate VQA accuracy on test samples."""
    
    correct = 0
    total = 0
    results_by_type = {}
    all_results = []
    
    for sample in tqdm(test_samples, desc="Evaluating"):
        try:
            result = query_system.query(
                image_path=sample['image_path'],
                question=sample['question'],
                k=k
            )
            
            predicted = result['predicted_answer'].lower().strip()
            ground_truth = sample['answer'].lower().strip()
            is_correct = predicted == ground_truth
            
            total += 1
            if is_correct:
                correct += 1
            
            # Track by question type
            qtype = sample['question_type']
            if qtype not in results_by_type:
                results_by_type[qtype] = {'correct': 0, 'total': 0}
            results_by_type[qtype]['total'] += 1
            if is_correct:
                results_by_type[qtype]['correct'] += 1
            
            all_results.append({
                'question': sample['question'],
                'predicted': predicted,
                'ground_truth': ground_truth,
                'correct': is_correct,
                'question_type': qtype,
                'confidence': result['confidence']
            })
            
        except Exception as e:
            print(f"\n⚠️ Error: {e}")
    
    # Calculate metrics
    accuracy = correct / total if total > 0 else 0
    
    type_accuracies = {}
    for qtype, data in results_by_type.items():
        type_accuracies[qtype] = data['correct'] / data['total'] if data['total'] > 0 else 0
    
    return {
        'overall_accuracy': accuracy,
        'correct': correct,
        'total': total,
        'accuracy_by_type': type_accuracies,
        'counts_by_type': results_by_type,
        'all_results': all_results
    }

print("✓ Evaluation functions ready")

In [ ]:
# Cell 26: Run Evaluation

# Evaluate on a subset (adjust size based on time constraints)
EVAL_SIZE = 500  # Number of samples to evaluate

# Random sample for evaluation
np.random.seed(42)
eval_indices = np.random.choice(len(samples), min(EVAL_SIZE, len(samples)), replace=False)
eval_samples = [samples[i] for i in eval_indices]

print(f"Evaluating on {len(eval_samples)} samples...\n")

eval_results = evaluate_vqa_accuracy(query_system, eval_samples, k=5)

print("\n" + "=" * 60)
print("📊 EVALUATION RESULTS")
print("=" * 60)
print(f"\n🎯 Overall Accuracy: {eval_results['overall_accuracy']:.1%}")
print(f"   ({eval_results['correct']}/{eval_results['total']} correct)")

print("\n📊 Accuracy by Question Type:")
for qtype, acc in sorted(eval_results['accuracy_by_type'].items(), 
                         key=lambda x: x[1], reverse=True):
    count = eval_results['counts_by_type'][qtype]['total']
    print(f"   {qtype:12s}: {acc:5.1%} ({count} samples)")

In [ ]:
# Cell 27: Analyze Errors

# Get incorrect predictions
errors = [r for r in eval_results['all_results'] if not r['correct']]

print("=" * 60)
print("🔍 ERROR ANALYSIS")
print("=" * 60)
print(f"\nTotal errors: {len(errors)}")

# Error distribution by type
error_types = Counter([e['question_type'] for e in errors])
print("\n📊 Errors by Question Type:")
for qtype, count in error_types.most_common():
    print(f"   {qtype:12s}: {count}")

# Show example errors
print("\n📝 Example Errors:")
for i, error in enumerate(errors[:5], 1):
    print(f"\n   [{i}] Type: {error['question_type']}")
    print(f"       Q: {error['question']}")
    print(f"       Predicted: {error['predicted']}")
    print(f"       Correct: {error['ground_truth']}")

In [ ]:
# Cell 28: Save Evaluation Results

eval_results_file = DATA_DIR / "evaluation_results.json"

# Save results (excluding large arrays)
results_to_save = {
    'overall_accuracy': eval_results['overall_accuracy'],
    'correct': eval_results['correct'],
    'total': eval_results['total'],
    'accuracy_by_type': eval_results['accuracy_by_type'],
    'counts_by_type': eval_results['counts_by_type'],
    'eval_size': len(eval_samples),
    'k': 5,
    'timestamp': datetime.now().isoformat()
}

with open(eval_results_file, 'w') as f:
    json.dump(results_to_save, f, indent=2)

print(f"✓ Results saved to {eval_results_file}")

---
# 🔬 Phase 8: CoAttention Comparison (Optional)

In [ ]:
# Cell 29: CoAttention Model Comparison
# Uncomment this cell to compare with your CoAttention model

# ============================================
# 🔬 COATTENTION COMPARISON
# ============================================
# Uncomment and modify to compare with your trained model

# from your_coattention_module import CoAttentionVQA
# 
# # Load your trained model
# coattention_model = CoAttentionVQA.load("path/to/model.pt")
# coattention_model.to(DEVICE)
# coattention_model.eval()
# 
# def compare_models(sample, vilt_result, coattention_model):
#     """Compare ViLT vector search with CoAttention model."""
#     
#     # Get CoAttention prediction
#     image = Image.open(sample['image_path']).convert('RGB')
#     coattention_answer = coattention_model.predict(image, sample['question'])
#     
#     ground_truth = sample['answer'].lower()
#     vilt_correct = vilt_result['predicted_answer'].lower() == ground_truth
#     coattention_correct = coattention_answer.lower() == ground_truth
#     
#     return {
#         'question': sample['question'],
#         'ground_truth': ground_truth,
#         'vilt_answer': vilt_result['predicted_answer'],
#         'vilt_correct': vilt_correct,
#         'coattention_answer': coattention_answer,
#         'coattention_correct': coattention_correct
#     }
# 
# # Run comparison
# comparison_results = []
# for sample in tqdm(eval_samples[:100]):
#     vilt_result = query_system.query(sample['image_path'], sample['question'])
#     comparison = compare_models(sample, vilt_result, coattention_model)
#     comparison_results.append(comparison)
# 
# # Calculate comparison metrics
# vilt_accuracy = sum(r['vilt_correct'] for r in comparison_results) / len(comparison_results)
# coattention_accuracy = sum(r['coattention_correct'] for r in comparison_results) / len(comparison_results)
# 
# print(f"ViLT Vector Search Accuracy: {vilt_accuracy:.1%}")
# print(f"CoAttention Model Accuracy: {coattention_accuracy:.1%}")

print("ℹ️ CoAttention comparison cell - uncomment to enable")

---
# 🧹 Phase 9: Utilities & Cleanup

In [ ]:
# Cell 30: Utility Functions

def get_collection_stats(collection):
    """Get collection statistics."""
    stats = {
        'total_documents': collection.count_documents({}),
        'question_types': {},
        'answer_types': {}
    }
    
    # Question type counts
    pipeline = [{"$group": {"_id": "$question_type", "count": {"$sum": 1}}}]
    for doc in collection.aggregate(pipeline):
        stats['question_types'][doc['_id']] = doc['count']
    
    # Answer type counts
    pipeline = [{"$group": {"_id": "$answer_type", "count": {"$sum": 1}}}]
    for doc in collection.aggregate(pipeline):
        stats['answer_types'][doc['_id']] = doc['count']
    
    return stats

def delete_all_documents(collection, confirm: bool = False):
    """Delete all documents from collection."""
    if not confirm:
        print("⚠️ Set confirm=True to actually delete")
        return
    
    result = collection.delete_many({})
    print(f"✓ Deleted {result.deleted_count} documents")

def export_embeddings(samples, embeddings, output_path: Path):
    """Export embeddings to file."""
    data = {
        'samples': samples,
        'embeddings': embeddings
    }
    with open(output_path, 'wb') as f:
        pickle.dump(data, f)
    print(f"✓ Exported to {output_path}")

# Display current stats
stats = get_collection_stats(cosmos_collection)
print("📊 Collection Statistics:")
print(f"   Total documents: {stats['total_documents']}")
print(f"   Question types: {len(stats['question_types'])}")

In [ ]:
# Cell 31: Cleanup (Run when done)

# ============================================
# 🧹 CLEANUP
# ============================================

# Close Cosmos DB connection
# cosmos_client.close()
# print("✓ Cosmos DB connection closed")

# Clear checkpoints
# checkpoint_mgr.clear()

# Delete cached embeddings
# if embeddings_cache_file.exists():
#     embeddings_cache_file.unlink()
#     print("✓ Embeddings cache deleted")

# Delete all documents (CAREFUL!)
# delete_all_documents(cosmos_collection, confirm=True)

# Clear GPU memory
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()
#     print("✓ GPU memory cleared")

print("""ℹ️ Cleanup options available:
- Uncomment cosmos_client.close() to close DB connection
- Uncomment checkpoint_mgr.clear() to clear checkpoints
- Uncomment delete_all_documents() to delete all embeddings
- Uncomment torch.cuda.empty_cache() to free GPU memory
""")

print("\n" + "=" * 60)
print("🎉 NOTEBOOK COMPLETE!")
print("=" * 60)
print(f"\n✅ Embeddings: {len(all_embeddings)} generated")
print(f"✅ Documents: {cosmos_collection.count_documents({})} in Cosmos DB")
print(f"✅ Evaluation: {eval_results['overall_accuracy']:.1%} accuracy")
print("\n🚀 Your VQA vector search system is ready to use!")